# 02 - Data Validation

**Objective:** Validate breast cancer records using Pydantic models and Pandera DataFrame schemas.

**Tools:** Pydantic (row-level), Pandera (DataFrame-level)

**Steps:**
1. Define a Pydantic model for breast cancer records
2. Implement row-level validation
3. Define a Pandera DataFrame schema
4. Implement DataFrame-level validation
5. Test on clean and corrupted data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from pydantic import BaseModel, field_validator, model_validator
import pandera as pa
from typing import List
from pandera.typing import DataFrame, Series

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
PROCESSED_DIR = Path("../data/processed")
CORRUPED_DIR = PROCESSED_DIR / "corrupted"
df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")

df = pd.read_csv(CORRUPED_DIR / "corrupted_outliers.csv")

col_names = ["Diagnosis"]
attributes_cont = [
    "Radius", "Texture", "Perimeter", "Area", "Smoothness",
    "Compactness", "Concavity", "ConcavePoints", "Symmetry", "FractalDimension",
]

for feat in attributes_cont:
    for stat in ["Mean", "SE", "Worst"]:
        col_names.append(f"{stat}_{feat}")


print(f"Data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Data shape: (569, 31)
Columns: ['Diagnosis', 'Mean_Radius', 'SE_Radius', 'Worst_Radius', 'Mean_Texture', 'SE_Texture', 'Worst_Texture', 'Mean_Perimeter', 'SE_Perimeter', 'Worst_Perimeter', 'Mean_Area', 'SE_Area', 'Worst_Area', 'Mean_Smoothness', 'SE_Smoothness', 'Worst_Smoothness', 'Mean_Compactness', 'SE_Compactness', 'Worst_Compactness', 'Mean_Concavity', 'SE_Concavity', 'Worst_Concavity', 'Mean_ConcavePoints', 'SE_ConcavePoints', 'Worst_ConcavePoints', 'Mean_Symmetry', 'SE_Symmetry', 'Worst_Symmetry', 'Mean_FractalDimension', 'SE_FractalDimension', 'Worst_FractalDimension']


In [3]:
df.describe()

,Diagnosis,Mean_Radius,SE_Radius,Worst_Radius,Mean_Texture,SE_Texture,Worst_Texture,Mean_Perimeter,SE_Perimeter,Worst_Perimeter,...,Worst_Concavity,Mean_ConcavePoints,SE_ConcavePoints,Worst_ConcavePoints,Mean_Symmetry,SE_Symmetry,Worst_Symmetry,Mean_FractalDimension,SE_FractalDimension,Worst_FractalDimension
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,0.372583,23.045543,19.289649,91.969033,1052.990158,0.096360,0.104341,0.164686,0.048919,0.181162,...,16.269190,42.128717,107.261213,880.583128,0.227021,0.254265,0.272188,0.193067,0.290076,0.083946
std,0.483918,43.530740,4.301036,24.298981,2248.487109,0.014064,0.052813,0.531768,0.038803,0.027414,...,4.833242,78.535978,33.602542,569.356993,0.454630,0.157336,0.208624,0.429184,0.061867,0.018061
min,0.000000,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,0.000000,11.760000,16.170000,75.170000,427.900000,0.086370,0.064920,0.029740,0.020310,0.161900,...,13.010000,21.320000,84.110000,515.300000,0.117200,0.147200,0.114500,0.065280,0.250400,0.071460
50%,0.000000,13.540000,18.840000,86.240000,565.400000,0.095870,0.092630,0.066510,0.033500,0.179200,...,14.970000,25.590000,97.660000,686.500000,0.132700,0.211900,0.226700,0.104500,0.282200,0.080040
75%,1.000000,16.690000,21.800000,104.100000,869.500000,0.105300,0.130400,0.144500,0.074000,0.195700,...,18.790000,30.760000,125.400000,1084.000000,0.148100,0.339100,0.382900,0.173200,0.317900,0.092080
max,1.000000,388.000000,39.280000,188.500000,28176.000000,0.163400,0.345400,6.828800,0.201200,0.304000,...,36.040000,570.240000,251.200000,4254.000000,3.004800,1.058000,1.252000,4.656000,0.663800,0.207500


In [31]:
df.describe()

,Diagnosis,Mean_Radius,SE_Radius,Worst_Radius,Mean_Texture,SE_Texture,Worst_Texture,Mean_Perimeter,SE_Perimeter,Worst_Perimeter,...,Worst_Concavity,Mean_ConcavePoints,SE_ConcavePoints,Worst_ConcavePoints,Mean_Symmetry,SE_Symmetry,Worst_Symmetry,Mean_FractalDimension,SE_FractalDimension,Worst_FractalDimension
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,0.372583,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,0.483918,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,0.000000,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,0.000000,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,0.000000,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,1.000000,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,1.000000,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [ ]:
df.columns[df[col_names].isnull().sum() > 0]

Index([], dtype='object')

### Pydantic Model Definition

Pydantic validates data at the **record level** — each row is checked individually.
Define a model with typed fields and add `@field_validator` decorators to enforce
business rules like valid diagnosis values, positive measurements, etc.

This catches row-level issues like a Diagnosis of 2 or a negative radius
that might slip through type-only checks.

In [ ]:
from dataclasses import dataclass

In [7]:
# === Executed Example: Pydantic Row-Level Validation ===
# This cell uses a small inline dataset (no CSV needed) to demonstrate
# how Pydantic catches invalid rows at the record level.

import pandas as pd
from pydantic import BaseModel, field_validator

data = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "diagnosis": [0, 1, 2, 0, 1],
    "radius": [14.5, 18.2, -3.0, 8.0, 999.0],
})
data

,id,diagnosis,radius
0,1,0,14.5
1,2,1,18.2
2,3,2,-3.0
3,4,0,8.0
4,5,1,999.0


In [12]:
from dataclasses import dataclass
import inspect

@dataclass(frozen = True,  order=True)
class RadiusValidator:
    id: int
    radius: float
    diagnosis: str


patient_diagnosis_benign = RadiusValidator(id=1, radius=14.5, diagnosis="B")  # Valid
print(patient_diagnosis_benign)
patient_diagnosis_malign = RadiusValidator(id=2, radius=-3.0, diagnosis="M")  # Invalid radius

print(patient_diagnosis_malign)

RadiusValidator(id=1, radius=14.5, diagnosis='B')
RadiusValidator(id=2, radius=-3.0, diagnosis='M')


In [13]:
sorted([patient_diagnosis_benign, patient_diagnosis_malign])

[RadiusValidator(id=1, radius=14.5, diagnosis='B'),
 RadiusValidator(id=2, radius=-3.0, diagnosis='M')]

In [16]:
inspect.getmembers(sorted([patient_diagnosis_benign, patient_diagnosis_malign]))

[('__add__', <method-wrapper '__add__' of list object at 0x7d300c1c1400>),
 ('__class__', list),
 ('__class_getitem__', <function list.__class_getitem__>),
 ('__contains__',
  <method-wrapper '__contains__' of list object at 0x7d300c1c1400>),
 ('__delattr__',
  <method-wrapper '__delattr__' of list object at 0x7d300c1c1400>),
 ('__delitem__',
  <method-wrapper '__delitem__' of list object at 0x7d300c1c1400>),
 ('__dir__', <function list.__dir__()>),
 ('__doc__',
  'Built-in mutable sequence.\n\nIf no argument is given, the constructor creates a new empty list.\nThe argument must be an iterable if specified.'),
 ('__eq__', <method-wrapper '__eq__' of list object at 0x7d300c1c1400>),
 ('__format__', <function list.__format__(format_spec, /)>),
 ('__ge__', <method-wrapper '__ge__' of list object at 0x7d300c1c1400>),
 ('__getattribute__',
  <method-wrapper '__getattribute__' of list object at 0x7d300c1c1400>),
 ('__getitem__', <function list.__getitem__>),
 ('__getstate__', <function list.

In [ ]:


class Record(BaseModel):
    id: int
    diagnosis: int
    radius: float



    @model_validator(mode="before") # This runs before field validators, so we can check for missing keys
    @classmethod
    def check_required_fields(cls, values):
        required_fields = {"id", "diagnosis", "radius"}
        missing = required_fields - values.keys()
        if missing:
            raise ValueError(f"Missing fields: {missing}")
        return values


    @field_validator("diagnosis")
    @classmethod
    def diagnosis_binary(cls, v):
        if v not in [0, 1]:
            raise ValueError(f"Diagnosis must be 0 or 1, got {v}")
        return v

    @field_validator("radius")
    @classmethod
    def radius_positive(cls, v):
        if not (0 < v <= 50):
            raise ValueError(f"Radius out of range: {v}")
        return v

    class Config: 
        extra = "forbid"  # Disallow extra fields not defined in the model
        allow_mutation = False  # Make instances immutable
        anystr_strip_whitespace = True  # Strip whitespace from strings


valid = []
for _, row in data.iterrows():
    try:
        Record(**row.to_dict())
        valid.append(True)
    except ValueError:
        valid.append(False)

print(f"Pydantic: {sum(valid)}/{len(valid)} rows valid")

Pydantic: 3/5 rows valid


In [5]:
# === Commented Template: Pydantic Row-Level Validation ===
# Copy, paste, and adapt to your own dataset. Uncomment to use.

# import pandas as pd
# from pydantic import BaseModel, field_validator

# data = pd.DataFrame({
#     "field_1": [val1, val2, val3],
#     "field_2": [val1, val2, val3],
# })

# class MyRecord(BaseModel):
#     field_1: int
#     field_2: float

#     @field_validator("field_1")
#     @classmethod
#     def rule_name(cls, v):
#         if not (lower <= v <= upper):
#             raise ValueError(f"Constraint description: {v}")
#         return v

# valid = []
# for _, row in data.iterrows():
#     try:
#         MyRecord(**row.to_dict())
#         valid.append(True)
#     except ValueError:
#         valid.append(False)

# print(f"Pydantic: {sum(valid)}/{len(valid)} rows valid")

In [ ]:
class BreastCancerRecord(BaseModel):
    """Pydantic model for a single breast cancer record."""

    Diagnosis: int
    Mean_Radius: float
    SE_Radius: float
    Worst_Radius: float
    Mean_Texture: float
    SE_Texture: float
    Worst_Texture: float
    Mean_Perimeter: float
    SE_Perimeter: float
    Worst_Perimeter: float
    Mean_Area: float
    SE_Area: float
    Worst_Area: float
    Mean_Smoothness: float
    SE_Smoothness: float
    Worst_Smoothness: float
    Mean_Compactness: float
    SE_Compactness: float
    Worst_Compactness: float
    Mean_Concavity: float
    SE_Concavity: float
    Worst_Concavity: float
    Mean_ConcavePoints: float
    SE_ConcavePoints: float
    Worst_ConcavePoints: float
    Mean_Symmetry: float
    SE_Symmetry: float
    Worst_Symmetry: float
    Mean_FractalDimension: float
    SE_FractalDimension: float
    Worst_FractalDimension: float

    @field_validator("Diagnosis")
    @classmethod
    def diagnosis_binary(cls, v):
        if v not in [0, 1]:
            raise ValueError(f"Diagnosis must be 0 or 1, got {v}")
        return v

    @field_validator("Mean_Area", "SE_Area", "Worst_Area")
    @classmethod
    def area_positive(cls, v):
        if v < 0:
            raise ValueError(f"Area must be positive: {v}")
        return v
    
    @model_validator(mode="after")
    def check_consistency(self, values):
        if self["Mean_Radius"] > self["Worst_Radius"]:
            raise ValueError("Mean_Radius cannot be greater than Worst_Radius")
        return values


In [ ]:
def validate_with_pydantic(df: pd.DataFrame) -> List[bool]:
    """Validate each row of the DataFrame using the Pydantic model.

    Returns a list of booleans where True means the row is valid.
    """
    valid = []
    for _, row in df.iterrows():
        try:
            BreastCancerRecord(**row.to_dict())
            valid.append(True)
        except Exception:
            valid.append(False)

    valid_count = sum(valid)
    invalid_count = len(valid) - valid_count
    print(f"Pydantic: {valid_count} valid, {invalid_count} invalid out of {len(valid)} rows")
    return valid


valid_flags = validate_with_pydantic(df)
print(f"Pass rate: {sum(valid_flags) / len(valid_flags):.1%}")

Pydantic: 549 valid, 20 invalid out of 569 rows
Pass rate: 96.5%


## Define your own Error Class

```python
class ValidationError(Exception):
    """Custom exception for validation errors."""

    def __init__(self, value: str, message: str):
        self.message = message
        self.value = value
        super().__init__(message)
        
```

### Pandera Schema Definition

Pandera validates data at the **DataFrame level** — instead of checking one row at a time,
it defines column-wide constraints (data types, value ranges, nullable rules, etc.).

This is more efficient for bulk validation and catches issues like a column
that unexpectedly contains strings instead of numbers.

In [18]:
# === Executed Example: Pandera DataFrame-Level Validation ===
# This cell uses the same inline dataset to demonstrate how Pandera
# validates all rows at once, collecting all failures.

import pandas as pd
import pandera.pandas as pa

data = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "diagnosis": [0, 1, 2, 0, 1],
    "radius": [14.5, 18.2, -3.0, 8.0, 999.0],
})
print(data)

   id  diagnosis  radius
0   1          0    14.5
1   2          1    18.2
2   3          2    -3.0
3   4          0     8.0
4   5          1   999.0


In [ ]:


Schema = pa.DataFrameSchema({
    "id": pa.Column(pa.Int, nullable=False),
    "diagnosis": pa.Column(pa.Int, checks=pa.Check.isin([0, 1])),
    "radius": pa.Column(pa.Float, checks=[pa.Check.greater_than(0), pa.Check.le(50)]),
})


try:
    Schema.validate(data, lazy=True) # don't stop at first error, collect all
    print("Pandera: all rows passed")
except pa.errors.SchemaErrors as e:
    print(f"Pandera: {len(e.failure_cases)} failures found")
    print(e.failure_cases[["index", "column", "check", "failure_case"]].head())

Pandera: 3 failures found
   index     column                      check  failure_case
0      2  diagnosis               isin([0, 1])           2.0
1      2     radius            greater_than(0)          -3.0
2      4     radius  less_than_or_equal_to(50)         999.0


In [ ]:

@pa.check_output(Schema, lazy=True)  # validate output of this function against the schema
def validate_with_pandera(df: DataFrame) -> DataFrame:
    """Validate the DataFrame using the defined Pandera schema."""
    return df  # In a real use case, you might do transformations here

try:
    validate_with_pandera(df)
    print("Pandera: all rows passed")
except pa.errors.SchemaErrors as e:
    print(f"Pandera: {len(e.failure_cases)} failures found")
    print(e.failure_cases[["index", "column", "check", "failure_case"]].head())

In [9]:
# === Commented Template: Pandera DataFrame-Level Validation ===
# Copy, paste, and adapt to your own dataset. Uncomment to use.

# import pandas as pd
# import pandera as pa

# data = pd.DataFrame({
#     "field_1": [val1, val2, val3],
#     "field_2": [val1, val2, val3],
# })

# MySchema = pa.DataFrameSchema({
#     "field_1": pa.Column(pa.Int, nullable=False),
#     "field_2": pa.Column(pa.Float, checks=[pa.Check.ge(min_val), pa.Check.le(max_val)]),
# })

# try:
#     MySchema.validate(data, lazy=True)
#     print("Pandera: all rows valid")
# except pa.errors.SchemaErrors as e:
#     print(f"Pandera: {len(e.failure_cases)} failure(s)")

In [24]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

feature_cols = [
    "Mean_Radius", "SE_Radius", "Worst_Radius",
    "Mean_Texture", "SE_Texture", "Worst_Texture",
    "Mean_Perimeter", "SE_Perimeter", "Worst_Perimeter",
    "Mean_Area", "SE_Area", "Worst_Area",
    "Mean_Smoothness", "SE_Smoothness", "Worst_Smoothness",
    "Mean_Compactness", "SE_Compactness", "Worst_Compactness",
    "Mean_Concavity", "SE_Concavity", "Worst_Concavity",
    "Mean_ConcavePoints", "SE_ConcavePoints", "Worst_ConcavePoints",
    "Mean_Symmetry", "SE_Symmetry", "Worst_Symmetry",
    "Mean_FractalDimension", "SE_FractalDimension", "Worst_FractalDimension",
]

BreastCancerSchema = pa.DataFrameSchema({
    "Diagnosis": pa.Column(pa.Int, checks=pa.Check.isin([0, 1])),
    **{
        col: pa.Column(pa.Float, checks=[
            pa.Check.greater_than_or_equal_to(0),
            pa.Check.less_than_or_equal_to(5000)
        ])
        for col in feature_cols
    },
    "Mean_ConcavePoints": pa.Column(pa.Float, checks=[
        pa.Check.greater_than_or_equal_to(0),
        pa.Check.less_than_or_equal_to(1)
    ]),

})

print("BreastCancerSchema defined with all 31 columns")

BreastCancerSchema defined with all 31 columns


In [28]:
## pandera infer schema

try:
    inferred_schema = pa.infer_schema(df)
    print("Inferred schema:")
    print(inferred_schema)
except Exception as e:
    print(f"Error inferring schema: {e}")

with open("breast_cancer_schema.py", "w") as f:
    f.write("import pandera as pa\n\n")
    f.write("BreastCancerSchema = ")
    f.write(inferred_schema.to_script())

Inferred schema:
<Schema DataFrameSchema(
    columns={
        'Diagnosis': <Schema Column(name=Diagnosis, type=DataType(int64))>
        'Mean_Radius': <Schema Column(name=Mean_Radius, type=DataType(float64))>
        'SE_Radius': <Schema Column(name=SE_Radius, type=DataType(float64))>
        'Worst_Radius': <Schema Column(name=Worst_Radius, type=DataType(float64))>
        'Mean_Texture': <Schema Column(name=Mean_Texture, type=DataType(float64))>
        'SE_Texture': <Schema Column(name=SE_Texture, type=DataType(float64))>
        'Worst_Texture': <Schema Column(name=Worst_Texture, type=DataType(float64))>
        'Mean_Perimeter': <Schema Column(name=Mean_Perimeter, type=DataType(float64))>
        'SE_Perimeter': <Schema Column(name=SE_Perimeter, type=DataType(float64))>
        'Worst_Perimeter': <Schema Column(name=Worst_Perimeter, type=DataType(float64))>
        'Mean_Area': <Schema Column(name=Mean_Area, type=DataType(float64))>
        'SE_Area': <Schema Column(name=SE_Are

In [11]:
def validate_with_pandera(df):
    """Validate the entire DataFrame using the Pandera schema."""
    try:
        BreastCancerSchema.validate(df, lazy=True)
        print("Pandera: all rows passed validation")
    except pa.errors.SchemaErrors as e:
        print(f"Pandera: {len(e.failure_cases)} failure(s) found")
        print(e.failure_cases[["index", "column", "check", "failure_case"]].head())


validate_with_pandera(df)

Pandera: all rows passed validation


### Pydantic With Pandera

In [12]:
# Load corrupted data with missing values in Diagnosis
df_corrupted = pd.read_csv(PROCESSED_DIR / "corrupted" / "corrupted_missing_light.csv")
print(f"Corrupted data shape: {df_corrupted.shape}")
print(f"Missing values per column:\n{df_corrupted.isnull().sum()}\n")

# Pydantic validation on corrupted data
print("--- Pydantic Validation ---")
valid_corrupted = validate_with_pydantic(df_corrupted)
cv = sum(valid_corrupted)
ct = len(valid_corrupted)
print(f"Corrupted pass rate: {cv}/{ct} ({cv/ct:.1%})")

# Pandera validation on corrupted data
print("\n--- Pandera Validation ---")
validate_with_pandera(df_corrupted)

print("\n" + "="*60 + "\n")

# Generate corrupted data programmatically using the injection module
# Load different corrupted variants from pre-generated files
df_outliers = pd.read_csv(PROCESSED_DIR / "corrupted" / "corrupted_outliers.csv")

print(f"Outlier data shape: {df_outliers.shape}")
print(f"Mean_Radius range: [{df_outliers['Mean_Radius'].min():.2f}, {df_outliers['Mean_Radius'].max():.2f}]\n")

print("--- Pydantic Validation (outliers) ---")
valid_outliers = validate_with_pydantic(df_outliers)
print(f"Pass rate: {sum(valid_outliers)}/{len(valid_outliers)} ({sum(valid_outliers)/len(valid_outliers):.1%})")

print("\n--- Pandera Validation (outliers) ---")
validate_with_pandera(df_outliers)

Corrupted data shape: (569, 31)
Missing values per column:
Diagnosis                 28
Mean_Radius                0
SE_Radius                  0
Worst_Radius               0
Mean_Texture               0
SE_Texture                 0
Worst_Texture              0
Mean_Perimeter             0
SE_Perimeter               0
Worst_Perimeter            0
Mean_Area                  0
SE_Area                    0
Worst_Area                 0
Mean_Smoothness            0
SE_Smoothness              0
Worst_Smoothness           0
Mean_Compactness           0
SE_Compactness             0
Worst_Compactness          0
Mean_Concavity             0
SE_Concavity               0
Worst_Concavity            0
Mean_ConcavePoints         0
SE_ConcavePoints           0
Worst_ConcavePoints        0
Mean_Symmetry              0
SE_Symmetry                0
Worst_Symmetry             0
Mean_FractalDimension      0
SE_FractalDimension        0
Worst_FractalDimension     0
dtype: int64

--- Pydantic Validation ---


### Exercises

1. **Add more validators**: Add a `@field_validator` for `Mean_Radius` that rejects values below 0 or above 30 (a realistic upper bound for breast cancer data).
2. **Compare error messages**: Run the same corrupted data through both Pydantic and Pandera — which one gives more informative error messages?
3. **Performance test**: Time both validators on the full dataset (use `%%timeit` in a cell) — is Pandera noticeably faster for bulk validation?
4. **Custom error handler**: Modify `validate_with_pydantic()` to collect and return the actual error messages (not just True/False) so students can see why a row failed.
5. **Schema evolution**: What happens if a new column is added to the CSV? Will Pandera reject it or pass it through? Check the `strict` parameter in `pa.DataFrameSchema`.